### Cell 1 — Vectors, Matrices, Determinant, Inverse (NumPy)

- **Mục tiêu:** Ôn nhanh đại số tuyến tính cho OLS.
- **Vector & phép toán:** dot (`v @ w`), chuẩn L2 (`||v||`).
- **Ma trận:** nhân ma trận–ma trận/ma trận–vector (`A @ B`, `Aᵀv`).
- **Định thức & nghịch đảo:**
  - `det(A)` đo “độ thể tích” → `det=0` ⇒ **không khả nghịch**.
  - `A⁻¹` chỉ tồn tại khi `det(A) ≠ 0`; kiểm tra `A @ A⁻¹ ≈ I`.
- **Demo singular:** ma trận tỷ lệ hàng/cột (`S`) có `det=0` → `np.linalg.inv(S)` báo lỗi (đúng kỳ vọng).


In [ ]:
# CELL 1 — Vectors, Matrices, Determinant, Inverse (NumPy)
import numpy as np

np.set_printoptions(precision=4, suppress=True)

# 1) Vectors (1D) & basic ops
v = np.array([1., 2., 3.])                # vector cột (khái niệm), dạng 1D
w = np.array([4., 5., 6.])
dot_vw = v @ w                             # tích vô hướng
norm_v  = np.linalg.norm(v)                # chuẩn L2

print("Vector v:", v)
print("Vector w:", w)
print("Dot(v,w):", dot_vw)
print("||v||:", norm_v)

# 2) Matrices (2D) & basic ops
A = np.array([[2., 1., 0.],
              [1., 3., 1.],
              [0., 1., 2.]])
B = np.array([[1., 0., 2.],
              [0., 1., 1.],
              [3., 1., 0.]])

AB   = A @ B
At_v = A.T @ v                             # nhân ma trận–vector

print("\nMatrix A:\n", A)
print("Matrix B:\n", B)
print("A @ B:\n", AB)
print("A^T @ v:", At_v)

# 3) Determinant (det)
detA = np.linalg.det(A)
print("\nDeterminant det(A):", detA)

# 4) Inverse (A^{-1}) — chỉ tồn tại khi det(A) != 0
if abs(detA) < 1e-12:
    print("A is singular (det ~ 0), no inverse.")
    A_inv = None
else:
    A_inv = np.linalg.inv(A)
    print("A^{-1}:\n", A_inv)
    # Kiểm tra A @ A^{-1} ≈ I
    I_approx = A @ A_inv
    print("A @ A^{-1} (≈ I):\n", I_approx)

# 5) Ví dụ ma trận suy biến (không khả nghịch)
S = np.array([[1., 2.],
              [2., 4.]])   # hàng 2 = 2 * hàng 1 → det = 0
detS = np.linalg.det(S)
print("\nSingular S:\n", S)
print("det(S):", detS)
try:
    S_inv = np.linalg.inv(S)
except np.linalg.LinAlgError as e:
    print("inv(S) error (expected):", e)


Vector v: [1. 2. 3.]
Vector w: [4. 5. 6.]
Dot(v,w): 32.0
||v||: 3.7416573867739413

Matrix A:
 [[2. 1. 0.]
 [1. 3. 1.]
 [0. 1. 2.]]
Matrix B:
 [[1. 0. 2.]
 [0. 1. 1.]
 [3. 1. 0.]]
A @ B:
 [[2. 1. 5.]
 [4. 4. 5.]
 [6. 3. 1.]]
A^T @ v: [ 4. 10.  8.]

Determinant det(A): 8.000000000000002
A^{-1}:
 [[ 0.625 -0.25   0.125]
 [-0.25   0.5   -0.25 ]
 [ 0.125 -0.25   0.625]]
A @ A^{-1} (≈ I):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Singular S:
 [[1. 2.]
 [2. 4.]]
det(S): 0.0
inv(S) error (expected): Singular matrix


### Cell 2 — OLS bằng đại số ma trận:  β̂ = (XᵀX)⁻¹Xᵀy

- **Bài toán:** Tối thiểu hoá `||y − Xβ||²` → **normal equations**: `XᵀX β = Xᵀy`.
- **Cách làm:**
  1) Thêm **intercept**: `Xb = [1, X]`.
  2) Tính `XtX = XbᵀXb`, `Xty = Xbᵀy`.
  3) Ước lượng **OLS**: `β̂ = inv(XtX) @ Xty`.
- **Đánh giá:** Tính `ŷ = Xbβ̂`, residual `r = y − ŷ`, `MSE`, `R²`.
- **Kết quả hợp lý:** `β_hat` gần `β_true` (vì có nhiễu), chứng tỏ phép suy diễn và cài đặt đúng.


In [ ]:
# CELL 2 — OLS bằng đại số ma trận: β̂ = (XᵀX)^{-1} Xᵀ y (NumPy, dùng inv)
import numpy as np

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# 1) Tạo dữ liệu giả lập (n mẫu, p đặc trưng)
n, p = 200, 3
X = rng.normal(size=(n, p))
true_beta = np.array([1.5, -2.0, 0.7, 3.0])  # [β0, β1, β2, β3]
eps = rng.normal(scale=0.5, size=n)

# Thêm intercept vào X
Xb = np.column_stack([np.ones(n), X])  # (n, p+1)
y = Xb @ true_beta + eps

# 2) OLS “thủ công” với numpy.linalg.inv
XtX = Xb.T @ Xb
Xty = Xb.T @ y
beta_hat = np.linalg.inv(XtX) @ Xty   # β̂ = (XᵀX)^{-1} Xᵀ y

# 3) Dự báo & metrics cơ bản
y_hat = Xb @ beta_hat
resid = y - y_hat
sse = np.sum(resid**2)
sst = np.sum((y - y.mean())**2)
mse = sse / n
r2  = 1 - sse/sst

print("β_true :", true_beta)
print("β_hat  :", beta_hat)
print(f"MSE    : {mse:.4f}")
print(f"R²     : {r2:.4f}")

# (Tuỳ chọn kiểm tra ổn định — chỉ để tham khảo, không bắt buộc đề)
# beta_chk, *_ = np.linalg.lstsq(Xb, y, rcond=None)
# print("β_lstsq:", beta_chk)


β_true : [ 1.5 -2.   0.7  3. ]
β_hat  : [ 1.4815 -1.9816  0.6962  2.9742]
MSE    : 0.2566
R²     : 0.9810


### Cell 3 — Sanity checks & edge cases cho OLS

- **Ổn định số học:** In `det(XᵀX)` và **điều kiện** `cond(XᵀX)`:
  - `det` lớn hơn 0, `cond` nhỏ → **khả nghịch & ổn định** khi dùng `inv`.
  - `det≈0` hoặc `cond` rất lớn → cảnh báo suy biến/đa cộng tuyến.
- **Kiểm tra lại β̂:** Gọi lại hàm OLS và so sánh với Cell 2 (kết quả khớp).
- **Singular demo:** Tạo đa cộng tuyến hoàn hảo (`x₂ = 2x₁`) → `XᵀX` **singular** → `inv` ném lỗi (đúng hành vi mong đợi).
- **Ghi chú:** Bài yêu cầu dùng `np.linalg.inv()`; thực tế có thể dùng `pinv`/`lstsq` khi `XᵀX` kém điều kiện.


In [ ]:
# CELL 3 — Sanity checks & edge cases cho OLS (det, cond, singular demo)

import numpy as np

def ols_via_inv(Xb: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Trả về β̂ = (XᵀX)^{-1} Xᵀy với kiểm tra cơ bản về khả nghịch."""
    XtX = Xb.T @ Xb
    det = np.linalg.det(XtX)
    cond = np.linalg.cond(XtX)
    print(f"det(X'X) = {det:.4e} | cond(X'X) = {cond:.2e}")
    if abs(det) < 1e-12:
        raise np.linalg.LinAlgError("X'X singular (det≈0) → không thể dùng inv.")
    beta_hat = np.linalg.inv(XtX) @ (Xb.T @ y)
    return beta_hat

# 1) Kiểm tra lại với bộ dữ liệu ở Cell 2 (đã có Xb, y)
beta_hat_chk = ols_via_inv(Xb, y)
print("β_hat (check) :", beta_hat_chk)

# 2) Demo singular: tạo đa cộng tuyến hoàn hảo → inv sẽ lỗi
n = 50
x1 = np.linspace(0, 1, n)
x2 = 2.0 * x1             # x2 = 2*x1 → cột phụ thuộc tuyến tính
X_sing = np.column_stack([np.ones(n), x1, x2])  # [1, x1, 2*x1] → X'X singular
y_sing = 3 + 4*x1 + np.random.normal(scale=0.1, size=n)

print("\n-- Singular demo --")
try:
    _ = ols_via_inv(X_sing, y_sing)
except np.linalg.LinAlgError as e:
    print("Expected error:", e)

# Ghi chú:
# - Với yêu cầu bài, ta dùng numpy.linalg.inv().
# - Khi X'X gần singular (det≈0, cond rất lớn), ước lượng bằng inv kém ổn định.
#   (Trong thực tế có thể dùng pinv/lstsq hoặc thêm regularization, nhưng KHÔNG thuộc phạm vi yêu cầu này.)


det(X'X) = 1.3253e+09 | cond(X'X) = 1.35e+00
β_hat (check) : [ 1.4815 -1.9816  0.6962  2.9742]

-- Singular demo --
det(X'X) = 0.0000e+00 | cond(X'X) = inf
Expected error: X'X singular (det≈0) → không thể dùng inv.
